In [48]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
from palettable.wesanderson import Darjeeling2_5
from prettytable import PrettyTable
from scipy.stats import chi2_contingency
from statsmodels.sandbox.regression.predstd import wls_prediction_std
from statsmodels.stats.proportion import proportions_ztest
import statsmodels.formula.api as smf
import statsmodels.api as sm
import matplotlib.pyplot as plt
import warnings
import pickle
import joblib
warnings.filterwarnings('ignore')

In [50]:
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.1)
    Q3 = df[column].quantile(0.9)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

def plot_regression_with_ci(df, group, color, label , model):

    df = df[df['Group'] == group]

    # Get prediction std deviation and intervals
    prstd, iv_l, iv_u = wls_prediction_std(model)
    
    # Plotting
    plt.plot(df['EVENT_ID'].sort_values().values, model.predict(df).loc[df.sort_values('EVENT_ID').index], 'k-', color=color, label=f'{label}')
    #plt.fill_between(df['EVENT_ID'].sort_values().values, iv_l.loc[df.sort_values('EVENT_ID').index], iv_u.loc[df.sort_values('EVENT_ID').index], color=color, alpha=0.05)

def data_availability_plot(data, cutoff, threshold=0.2):
    # Seaborn theme and Wes Anderson palette
    sns.set_theme(style="whitegrid", context="paper")
    sns.set_palette(Darjeeling2_5.mpl_colors)

    fig, (ax1, ax2) = plt.subplots(figsize=(10, 6), ncols=2)

    # Left panel: data availability
    df_plot = pd.DataFrame(
        data.groupby('Group').count().values / data.groupby('Group').size().values.reshape(-1, 1),
        index=[0, 1, 2],
        columns=data.columns[:-1]
    )
    sns.lineplot(data=df_plot.T, ax=ax1)

    ax1.set_xticks(range(len(df_plot.columns)))
    ax1.set_xticklabels(df_plot.columns, rotation=45, ha='right')

    # Ensure tick labels resolved before reading them
    fig.canvas.draw()

    x_ticks = ax1.get_xticks()
    xtick_texts = np.array([t.get_text() for t in ax1.get_xticklabels()])
    ax1.hlines(threshold, x_ticks.min(), x_ticks.max(),
               label='Missingness Cutoff', linestyle='--', color='k')
    ax1.set_title('Data Availability')
    ax1.set_ylabel('Proportion of Samples with Data Available')
    ax1.set_xlabel('Visit Number')

    # Legend: append ' Group' to line labels, keep cutoff label unchanged
    handles, labels = ax1.get_legend_handles_labels()
    labels = [f"Group {int(float(l))}" if l not in ('Missingness Cutoff', '') else l for l in labels]
    ax1.legend(handles, labels)

    # Right panel: subgroup averages
    df_mean = pd.DataFrame(data.groupby('Group').mean()).T
    sns.lineplot(data=df_mean, ax=ax2)

    ax2.set_xticks(range(len(df_plot.columns)))
    ax2.set_xticklabels(df_plot.columns, rotation=45, ha='right')

    y_ticks = ax2.get_yticks()
    ax2.vlines(np.where(xtick_texts == cutoff)[0][0],
               y_ticks.min(), y_ticks.max(),
               label='Timepoint Cutoff', linestyle='--', color='k')
    ax2.set_title('Group Mean Z-Score')
    ax2.set_ylabel('Mean Z-Score')
    ax2.set_xlabel('Visit Number')

    # Legend: append ' Group' to line labels, keep cutoff label unchanged
    handles, labels = ax2.get_legend_handles_labels()
    labels = [f"Group {int(float(l))}" if l not in ('Timepoint Cutoff', '') else l for l in labels]
    ax2.legend(handles, labels)

    sns.despine(fig=fig)
    plt.tight_layout()
    plt.show()
    plt.clf

def plot_score_distributions(df_long) : 
    # Seaborn theme and Wes Anderson palette
    sns.set_theme(style="whitegrid", context="paper")
    
    # Prepare data in long form for seaborn
    df_plot = df_long[['Status', 'Status_resid']].rename(
        columns={'Status': 'Raw', 'Status_resid': 'Residual of Sex + Age'}
    )
    df_melt = df_plot.melt(var_name='Measure', value_name='Mean Z-Score').dropna()
    
    # Ensure common bin range across both distributions
    vals = df_melt['Mean Z-Score'].to_numpy()
    vals = vals[np.isfinite(vals)]
    binrange = (np.min(vals), np.max(vals))
    
    # Choose two colors from Darjeeling2_5
    palette = Darjeeling2_5.mpl_colors
    pal_map = {'Raw': palette[0], 'Residual of Sex + Age': palette[3]}
    
    plt.figure(figsize=(9, 5))
    
    # Histograms (matched bins, density-scaled)
    sns.histplot(
        data=df_melt,
        x='Mean Z-Score',
        hue='Measure',
        palette=pal_map,
        bins='fd',
        binrange=binrange,
        stat='density',
        common_norm=False,
        element='poly',
        alpha=0.25,
        edgecolor='white',
        linewidth=0.5
    )
    
    # KDE overlays (no extra legend entries)
    sns.kdeplot(
        data=df_melt,
        x='Mean Z-Score',
        hue='Measure',
        palette=pal_map,
        common_norm=False,
        lw=2,
        legend=True,
        clip=binrange
    )
    
    # Reference line at zero (useful for z-scores)
    plt.axvline(0, color='k', lw=1, ls='--', alpha=0.6)
    
    # Labels and legend
    plt.xlabel('Mean Z-Score')
    plt.ylabel('Density')
    plt.title('Distribution of Status vs Residual (Sex + Age)')
    #handles, labels = plt.gca().get_legend_handles_labels()
    #plt.legend(handles, labels, title='', frameon=False, loc='upper right')
    
    sns.despine()
    plt.tight_layout()
    plt.show()
    plt.clf()

def data_process(data, subgroups , events , events_df, print_final_event=True) : 
    events_filt = sorted(list(set(events) & set(data.columns)))
    if print_final_event : 
        print(f'final event : {events_filt[-1]}')
    
    df_long = data.reset_index().melt(id_vars=['Group' ,'PATNO'], var_name='EVENT_ID', value_name='Status')
    
    df_long = pd.merge(df_long , subgroups.reset_index() , left_on = ['PATNO' , 'EVENT_ID' , 'Group'],right_on =['PATNO' , 'EVENT_ID' , 'Group'] , how='outer')
    
    df_long = pd.merge(df_long , events_df , on =['PATNO' , 'EVENT_ID'] , how='outer')
    
    df_long = df_long[df_long['EVENT_ID'].isin(events_filt)]
    
    # Combined linear model, including interaction between predictor and group
    df_long['Group'] = pd.Categorical(df_long['Group'], categories=[0,1,2], ordered=True)

    return df_long

def ols_model_plots(model) : 
    fitted_vals = model.predict()
    residuals = model.resid
    
    fig, (ax1,ax2) = plt.subplots(figsize=(10, 6) , ncols=2)
    ax1.scatter(fitted_vals, residuals, alpha=0.5)
    ax1.hline(0, color='red', linestyle='--')
    ax1.set_xlabel('Fitted Values')
    ax1.set_ylabel('Residuals')
    ax1.set_title('Residuals vs Fitted Values')
    
    sm.qqplot(residuals, line='s', ax=ax2)  # 's' indicates standardized line
    ax2.set_title('Q-Q Plot of Residuals')
    plt.show()

def fa(model, labels , data , part) : 

    keys = [f'Factor_{i}' for i in range(1 , len(labels.keys())+1)]
    columns = [labels[key] for key in keys]

    data_cols = [col for col in data if col not in ['EVENT_ID' , 'PATNO'] and col[-3:] != 'TOT']
    factor_scores = model.transform(data[data_cols].dropna())  # Transform the original data set
    # Create a DataFrame from the scores
    factor_scores_df = pd.DataFrame(factor_scores, columns=columns).add_prefix(f'MDS_UPDRS_P{part}_')
    
    factor_scores_df['PATNO'] = data.dropna(subset=data_cols).reset_index(drop=True)['PATNO']
    factor_scores_df['EVENT_ID'] = data.dropna(subset=data_cols).reset_index(drop=True)['EVENT_ID']
    
    return factor_scores_df

def gen_factors(model = 'PPMI' , data='PPMI' , state='OFF' , model_state_off = False) : 
    with open(f'./../../data/02_processed/{data}/P1_MDSUPDRS.pkl' , 'rb') as file : 
        p1 = pickle.load(file)

    with open(f'./../../data/02_processed/{data}/P2_MDSUPDRS.pkl' , 'rb') as file : 
        p2 = pickle.load(file)
    
    with open(f'./../../data/02_processed/{data}/P3{state}_MDSUPDRS.pkl' , 'rb') as file : 
        p3 = pickle.load(file)

    model_p1 = joblib.load(f"./../../data/03_final/{model}_FA_model_P1.joblib")
    model_p2 = joblib.load(f"./../../data/03_final/{model}_FA_model_P2.joblib")
    if (state == 'ON' or state == 'on') and model_state_off : 
        model_p3 = joblib.load(f"./../../data/03_final/{model}_FA_model_P3OFF.joblib")
    else : 
        model_p3 = joblib.load(f"./../../data/03_final/{model}_FA_model_P3{state}.joblib")

    p1_fa = fa(model_p1['fa'], model_p1['labels'] , p1 , '1')
    p2_fa = fa(model_p2['fa'], model_p2['labels'] , p2 , '2')
    p3_fa = fa(model_p3['fa'], model_p3['labels'] , p3 , '3')

    if data == 'PPMI' : 
        subgroups = pd.read_csv('./../../data/02_processed/PPMI/subgroups.csv', index_col=0)
        events = subgroups['EVENT_ID'].unique()
        x_list = pd.DataFrame(np.arange(0 , 9.5 , 0.5) , index = events , columns=['trajectory'])
        events_df = x_list.reset_index()
        events_df.rename(columns={'index': 'EVENT_ID'}, inplace=True)
        
    elif data == 'OPDC' : 
        subgroups = pd.read_csv('./../../data/02_processed/OPDC/subgroups.csv')
        subgroups['EVENT_ID'] = subgroups['EVENT_ID'].astype(str)
        events = subgroups['EVENT_ID'].unique()
        x_list = pd.DataFrame(np.arange(0 , 9*1.5 , 1.5) , index = events , columns=['trajectory'])
        events_df = x_list.reset_index()
        events_df.rename(columns={'index': 'EVENT_ID'}, inplace=True)

    elif data == 'TRACKING' : 
        subgroups = pd.read_csv('./../../data/02_processed/TRACKING/subgroups.csv' , index_col=0)
        subgroups['EVENT_ID'] = subgroups['EVENT_ID'].astype(str)
        events = ['1','4','7','9','10','11']
        x_list = pd.DataFrame([0,1.5,3,4.5,6,7.5] , index = events , columns=['trajectory'])
        events_df = x_list.reset_index()
        events_df.rename(columns={'index': 'EVENT_ID'}, inplace=True)

    subgroups = subgroups.merge(events_df , on=['EVENT_ID'])
    events_df = subgroups[['PATNO' , 'EVENT_ID', 'trajectory']]
    subgroups.drop(columns='trajectory',inplace=True)

    return pd.merge((pd.merge(p1_fa , p2_fa , on = ['PATNO' , 'EVENT_ID'])), p3_fa , on = ['PATNO' , 'EVENT_ID']) , subgroups, events, events_df

In [360]:
def has_significant_trajectory(
    pthresh=0.05,
    state='ON',
    data='TRACKING',
    model='TRACKING',
    model_state_off=True,
    factor_col='MDS_UPDRS_P3_Factor_4',
    thresh=0.3,
    print_final_event=False,
    print_model_summary=False
):
    """
    Runs the provided pipeline and returns True if more than one 'trajectory' term
    (main or interaction) has p < pthresh in the OLS model on Status_resid.
    
    Depends on:
      - gen_factors(state, data, model, model_state_off)
      - data_process(wide_df, subgroups, events, events_df, print_final_event)
    """
    # Get core inputs
    df, subgroups, events, events_df = gen_factors(
        state=state, data=data, model=model, model_state_off=model_state_off
    )

    # Build wide matrix
    df = df.copy()
    if len(factor_col) > 1 :
        df['combo'] = df.loc[: , factor_col].sum(axis=1)
    else : 
        df['combo'] = df[factor_col[0]]
    wide = df.pivot_table(values='combo', columns='EVENT_ID', index='PATNO', observed=False)
    wide = pd.merge(
        wide.reset_index(),
        subgroups.reset_index()[['PATNO', 'Group']].drop_duplicates(),
        on='PATNO'
    ).set_index('PATNO')

    # Event filtering
    events = list(set(events) & set(wide.columns))
    events = np.array(events)[wide[events].isna().sum() / wide.shape[0] <= 1 - thresh]

    if len(events) > 1 :
    
        # Long format and cleanup
        df_long = data_process(wide, subgroups, events, events_df, print_final_event=print_final_event)
        df_long.dropna(subset=['Status'], inplace=True)
    
        # Categorical group
        df_long['Group'] = pd.Categorical(df_long['Group'], categories=[0, 1, 2], ordered=True)
    
        # Residualize Status and z-score residuals
        model_resid = smf.ols('Status ~ AGE_AT_VISIT + GENDER', data=df_long).fit()
        df_long['Status_resid'] = (model_resid.resid - model_resid.resid.mean()) / model_resid.resid.std()
    
        # Main model
        model_0 = smf.ols('Status_resid ~ trajectory * C(Group)', data=df_long).fit()
        if print_model_summary : 
            print(model_0.summary())
    
        # Extract p-values for trajectory terms
        tbl = model_0.summary2().tables[1]
        p_traj_from_tbl = tbl.loc[tbl.index.str.contains(r'\btrajectory:C\b'), 'P>|t|']

        a_v_b = model_0.t_test('trajectory:C(Group)[T.1] = 0').pvalue < pthresh
        a_v_c = model_0.t_test('trajectory:C(Group)[T.2] = 0').pvalue < pthresh
        b_v_c = model_0.t_test('trajectory:C(Group)[T.1] = trajectory:C(Group)[T.2]').pvalue < pthresh
        
        # Return the original boolean check
        return [a_v_b, a_v_c, b_v_c]
    else :
        return [np.False_, np.False_, np.False_]

def contrasts_to_wide(d, test='AvB'):
    """
    Convert nested dict:
      data -> FA_model -> state -> { (measure,): { 'AvB': bool, 'AvC': bool, 'BvC': bool }, ... }
    into a wide DataFrame with columns:
      ['data', 'FA_model', 'state', 'test', <one column per measure>]
    Values are booleans for the chosen `test`.
    """
    rows = []
    for data_key, fa_models in d.items():
        for fa_key, states in fa_models.items():
            for state_key, measures in states.items():
                row = {'data': data_key, 'FA_model': fa_key, 'state': state_key, 'test': test}
                for m_key, tests in measures.items():
                    # Unpack tuple keys like ('MDS_UPDRS_P1_DEPRESSION+ANXIETY',)
                    col = m_key[0] if isinstance(m_key, tuple) and len(m_key) == 1 else m_key
                    val = tests.get(test, np.nan)
                    # Ensure Python bools (handles numpy.bool_)
                    row[col] = bool(val) if isinstance(val, (bool, np.bool_)) else np.nan
                rows.append(row)
    return pd.DataFrame(rows)

In [361]:
has_significant_trajectory(
    pthresh=0.05,
    state = 'ON',
    data = 'OPDC',
    model = 'OPDC',
    model_state_off=False,
    factor_col=('MDS_UPDRS_P1_DEPRESSION+ANXIETY', 'MDS_UPDRS_P1_FATIGUE'),
    print_model_summary=True
)

                            OLS Regression Results                            
Dep. Variable:           Status_resid   R-squared:                       0.023
Model:                            OLS   Adj. R-squared:                  0.021
Method:                 Least Squares   F-statistic:                     10.67
Date:                Thu, 02 Apr 2026   Prob (F-statistic):           3.79e-10
Time:                        08:21:45   Log-Likelihood:                -3157.2
No. Observations:                2244   AIC:                             6326.
Df Residuals:                    2238   BIC:                             6361.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

[np.False_, np.False_, np.False_]

In [362]:
mds_updrs_cols = ['MDS_UPDRS_P1_DEPRESSION+ANXIETY', 'MDS_UPDRS_P1_FATIGUE',
       'MDS_UPDRS_P1_COGNITION', 'MDS_UPDRS_P2_EAT',
       'MDS_UPDRS_P2_RISE+WALK', 'MDS_UPDRS_P2_SPEECH', 'MDS_UPDRS_P3_TREMOR',
       'MDS_UPDRS_P3_PIG', 'MDS_UPDRS_P3_TAPPING_L', 'MDS_UPDRS_P3_TAPPING_R',
       'MDS_UPDRS_P3_NA', 'MDS_UPDRS_P3_RISING']
datasets = ['PPMI' , 'TRACKING' , 'OPDC']
FA_models = ['PPMI' , 'TRACKING' , 'OPDC']
states = ['ON' , 'OFF']

In [363]:
import itertools

In [364]:
res = {}
for dat in datasets : 
    res[dat] = {}
    for fa_mod in FA_models : 
        res[dat][fa_mod] = {}
        for stat in states : 
            res[dat][fa_mod][stat] = {}
            for i in range(len(mds_updrs_cols)) : 
                print(i)
                for mds_updrs in itertools.combinations(mds_updrs_cols , i+1) : 
                    sig_res = False
                    sig_res = has_significant_trajectory(
                                            pthresh=0.05,
                                            state = stat,
                                            data = dat,
                                            model = fa_mod,
                                            model_state_off=False,
                                            factor_col=mds_updrs
                                        )
                    res[dat][fa_mod][stat][mds_updrs] = {}
                    res[dat][fa_mod][stat][mds_updrs]['AvB'] = sig_res[0]
                    res[dat][fa_mod][stat][mds_updrs]['AvC'] = sig_res[1]
                    res[dat][fa_mod][stat][mds_updrs]['BvC'] = sig_res[2]

0
1
2
3
4
5
6
7
8
9
10
11
0
1
2
3
4
5
6
7
8
9
10
11
0
1
2
3
4
5
6
7
8
9
10
11
0
1
2
3
4
5
6
7
8
9
10
11
0
1
2
3
4
5
6
7
8
9
10
11
0
1
2
3
4
5
6
7
8
9
10
11
0
1
2
3
4
5
6
7
8
9
10
11
0
1
2
3
4
5
6
7
8
9
10
11
0
1
2
3
4
5
6
7
8
9
10
11
0
1
2
3
4
5
6
7
8
9
10
11
0
1
2
3
4
5
6
7
8
9
10
11
0
1
2
3
4
5
6
7
8
9
10
11
0
1
2
3
4
5
6
7
8
9
10
11
0
1
2
3
4
5
6
7
8
9
10
11
0
1
2
3
4
5
6
7
8
9
10
11
0
1
2
3
4
5
6
7
8
9
10
11
0
1
2
3
4
5
6
7
8
9
10
11
0
1
2
3
4
5
6
7
8
9
10
11


In [365]:
res_df = contrasts_to_wide(res , test='AvB')
res_df2 = contrasts_to_wide(res , test='AvC')
res_df3 = contrasts_to_wide(res , test='BvC')

res_all = pd.concat([res_df , res_df2 , res_df3])

In [366]:
columns = []
columns_keep = []
for col in res_all.columns : 
    if type(col) == str:
        col = col
    else : 
        col = ' + '.join(col)

    if 'nan' in col : 
        col = col.replace(' + nan', '')
    if 'RISE+WALK' in col : 
        col = col.replace('RISE+WALK' , 'RISE/WALK')
    columns += [col]
    if '_NA' in col :
        pass
    else : 
        columns_keep += [col]

res_all.columns = columns

In [367]:
res_all = res_all[columns_keep]

In [368]:
res_all

,data,FA_model,state,test,MDS_UPDRS_P1_DEPRESSION+ANXIETY,MDS_UPDRS_P1_FATIGUE,MDS_UPDRS_P1_COGNITION,MDS_UPDRS_P2_EAT,MDS_UPDRS_P2_RISE/WALK,MDS_UPDRS_P2_SPEECH,...,MDS_UPDRS_P1_DEPRESSION+ANXIETY + MDS_UPDRS_P1_FATIGUE + MDS_UPDRS_P1_COGNITION + MDS_UPDRS_P2_EAT + MDS_UPDRS_P2_RISE/WALK + MDS_UPDRS_P2_SPEECH + MDS_UPDRS_P3_TREMOR + MDS_UPDRS_P3_PIG + MDS_UPDRS_P3_TAPPING_R + MDS_UPDRS_P3_RISING,MDS_UPDRS_P1_DEPRESSION+ANXIETY + MDS_UPDRS_P1_FATIGUE + MDS_UPDRS_P1_COGNITION + MDS_UPDRS_P2_EAT + MDS_UPDRS_P2_RISE/WALK + MDS_UPDRS_P2_SPEECH + MDS_UPDRS_P3_TREMOR + MDS_UPDRS_P3_TAPPING_L + MDS_UPDRS_P3_TAPPING_R + MDS_UPDRS_P3_RISING,MDS_UPDRS_P1_DEPRESSION+ANXIETY + MDS_UPDRS_P1_FATIGUE + MDS_UPDRS_P1_COGNITION + MDS_UPDRS_P2_EAT + MDS_UPDRS_P2_RISE/WALK + MDS_UPDRS_P2_SPEECH + MDS_UPDRS_P3_PIG + MDS_UPDRS_P3_TAPPING_L + MDS_UPDRS_P3_TAPPING_R + MDS_UPDRS_P3_RISING,MDS_UPDRS_P1_DEPRESSION+ANXIETY + MDS_UPDRS_P1_FATIGUE + MDS_UPDRS_P1_COGNITION + MDS_UPDRS_P2_EAT + MDS_UPDRS_P2_RISE/WALK + MDS_UPDRS_P3_TREMOR + MDS_UPDRS_P3_PIG + MDS_UPDRS_P3_TAPPING_L + MDS_UPDRS_P3_TAPPING_R + MDS_UPDRS_P3_RISING,MDS_UPDRS_P1_DEPRESSION+ANXIETY + MDS_UPDRS_P1_FATIGUE + MDS_UPDRS_P1_COGNITION + MDS_UPDRS_P2_EAT + MDS_UPDRS_P2_SPEECH + MDS_UPDRS_P3_TREMOR + MDS_UPDRS_P3_PIG + MDS_UPDRS_P3_TAPPING_L + MDS_UPDRS_P3_TAPPING_R + MDS_UPDRS_P3_RISING,MDS_UPDRS_P1_DEPRESSION+ANXIETY + MDS_UPDRS_P1_FATIGUE + MDS_UPDRS_P1_COGNITION + MDS_UPDRS_P2_RISE/WALK + MDS_UPDRS_P2_SPEECH + MDS_UPDRS_P3_TREMOR + MDS_UPDRS_P3_PIG + MDS_UPDRS_P3_TAPPING_L + MDS_UPDRS_P3_TAPPING_R + MDS_UPDRS_P3_RISING,MDS_UPDRS_P1_DEPRESSION+ANXIETY + MDS_UPDRS_P1_FATIGUE + MDS_UPDRS_P2_EAT + MDS_UPDRS_P2_RISE/WALK + MDS_UPDRS_P2_SPEECH + MDS_UPDRS_P3_TREMOR + MDS_UPDRS_P3_PIG + MDS_UPDRS_P3_TAPPING_L + MDS_UPDRS_P3_TAPPING_R + MDS_UPDRS_P3_RISING,MDS_UPDRS_P1_DEPRESSION+ANXIETY + MDS_UPDRS_P1_COGNITION + MDS_UPDRS_P2_EAT + MDS_UPDRS_P2_RISE/WALK + MDS_UPDRS_P2_SPEECH + MDS_UPDRS_P3_TREMOR + MDS_UPDRS_P3_PIG + MDS_UPDRS_P3_TAPPING_L + MDS_UPDRS_P3_TAPPING_R + MDS_UPDRS_P3_RISING,MDS_UPDRS_P1_FATIGUE + MDS_UPDRS_P1_COGNITION + MDS_UPDRS_P2_EAT + MDS_UPDRS_P2_RISE/WALK + MDS_UPDRS_P2_SPEECH + MDS_UPDRS_P3_TREMOR + MDS_UPDRS_P3_PIG + MDS_UPDRS_P3_TAPPING_L + MDS_UPDRS_P3_TAPPING_R + MDS_UPDRS_P3_RISING,MDS_UPDRS_P1_DEPRESSION+ANXIETY + MDS_UPDRS_P1_FATIGUE + MDS_UPDRS_P1_COGNITION + MDS_UPDRS_P2_EAT + MDS_UPDRS_P2_RISE/WALK + MDS_UPDRS_P2_SPEECH + MDS_UPDRS_P3_TREMOR + MDS_UPDRS_P3_PIG + MDS_UPDRS_P3_TAPPING_L + MDS_UPDRS_P3_TAPPING_R + MDS_UPDRS_P3_RISING
0,PPMI,PPMI,ON,AvB,True,False,False,False,False,False,...,True,True,True,True,True,True,True,True,True,True
1,PPMI,PPMI,OFF,AvB,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,PPMI,TRACKING,ON,AvB,False,False,True,False,False,False,...,True,True,True,True,True,True,True,True,True,True
3,PPMI,TRACKING,OFF,AvB,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,PPMI,OPDC,ON,AvB,True,False,False,False,False,False,...,True,True,True,True,True,True,True,True,True,True
5,PPMI,OPDC,OFF,AvB,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
6,TRACKING,PPMI,ON,AvB,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
7,TRACKING,PPMI,OFF,AvB,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
8,TRACKING,TRACKING,ON,AvB,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
9,TRACKING,TRACKING,OFF,AvB,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [369]:
res_all.loc[: , ['data' , 'FA_model', 'state','test'] + list(res_all.iloc[:,4:].sum(axis=0).sort_values(ascending=False).index)].to_csv('results.csv')

In [374]:
res_all.loc[: , ['data' , 'FA_model', 'state','test'] + list(res_all.iloc[:,4:].sum(axis=0).sort_values(ascending=False).index)].iloc[: , :5]

,data,FA_model,state,test,MDS_UPDRS_P1_COGNITION + MDS_UPDRS_P3_TAPPING_L + MDS_UPDRS_P3_TAPPING_R + MDS_UPDRS_P3_RISING
0,PPMI,PPMI,ON,AvB,True
1,PPMI,PPMI,OFF,AvB,False
2,PPMI,TRACKING,ON,AvB,True
3,PPMI,TRACKING,OFF,AvB,False
4,PPMI,OPDC,ON,AvB,True
5,PPMI,OPDC,OFF,AvB,False
6,TRACKING,PPMI,ON,AvB,False
7,TRACKING,PPMI,OFF,AvB,False
8,TRACKING,TRACKING,ON,AvB,False
9,TRACKING,TRACKING,OFF,AvB,False


In [183]:
results = pd.read_csv('results.csv')

In [377]:
import pandas as pd

def find_common_true_measures(
    df,
    dataset_col='data',
    key_cols=('FA_model', 'state', 'test'),
    prefixes=('MDS-UPDRS', 'MDS_UPDRS'),
    min_datasets=2
):
    """
    Find measure columns whose names start with any of `prefixes` that have a
    common True across at least `min_datasets` distinct datasets, for the
    same (FA_model, state, test) combination.

    Returns:
      hits: DataFrame with columns [FA_model, state, test, measure, n_datasets_true]
            for the combos that meet the threshold.
      measures: sorted list of measure names that meet the rule for at least one combo.
    """
    keys = list(key_cols)

    # Pick only measure columns with the desired prefixes
    def _starts_with_prefix(c):
        return isinstance(c, str) and any(c.startswith(p) for p in prefixes)

    exclude = set([dataset_col] + keys)
    measure_cols = [c for c in df.columns if c not in exclude and _starts_with_prefix(c)]
    if not measure_cols:
        raise ValueError("No measure columns found with the given prefixes.")

    # Coerce to boolean and treat NaN as False
    work = df[[dataset_col] + keys + measure_cols].copy()
    for c in measure_cols:
        work[c] = work[c].astype('boolean').fillna(False)

    # Long form
    long = work.melt(
        id_vars=[dataset_col] + keys,
        value_vars=measure_cols,
        var_name='measure',
        value_name='value'
    )

    # Collapse duplicates within a dataset: any True per (keys, dataset, measure)
    per_ds = (
        long.groupby(keys + [dataset_col, 'measure'], dropna=False)['value']
            .any()
            .reset_index()
    )

    # Count how many datasets are True per (keys, measure)
    counts = (
        per_ds.groupby(keys + ['measure'], dropna=False)['value']
              .sum()
              .reset_index(name='n_datasets_true')
    )

    # Keep only those with at least min_datasets True datasets
    hits = (
        counts[counts['n_datasets_true'] >= min_datasets]
        .sort_values(keys + ['measure'])
        .reset_index(drop=True)
    )

    measures = sorted(hits['measure'].unique())
    return hits, measures


In [404]:
hits, measures = find_common_true_measures(res_all[np.logical_and(res_all['test'].isin(['AvB','AvC']) , res_all['state'] == 'ON')])

In [419]:
factor = measures[13]
res_all[res_all[factor] == True].loc[:,['data','FA_model','state','test',factor]]

,data,FA_model,state,test,MDS_UPDRS_P1_COGNITION + MDS_UPDRS_P3_PIG
1,PPMI,PPMI,OFF,AvB,True
2,PPMI,TRACKING,ON,AvB,True
3,PPMI,TRACKING,OFF,AvB,True
5,PPMI,OPDC,OFF,AvB,True
12,OPDC,PPMI,ON,AvB,True
14,OPDC,TRACKING,ON,AvB,True
16,OPDC,OPDC,ON,AvB,True
5,PPMI,OPDC,OFF,AvC,True
12,OPDC,PPMI,ON,BvC,True
14,OPDC,TRACKING,ON,BvC,True
